# 🎙️ Lumini AI Studio - Free VoxCPM2 Voice Cloning Server (Google Colab GPU)
### 100% Free Zero-Shot Voice Cloning & 48kHz Studio Quality Speech Engine

---
### 📌 အသုံးပြုနည်းလမ်းညွှန်:
1. `Runtime -> Change runtime type -> T4 GPU` ရွေးထားကြောင်း စစ်ဆေးပါ။
2. အောက်ပါ Cell များကို **Play (Run)** နှိပ်ပါ။
3. ထွက်လာသော `https://xxxx.trycloudflare.com` URL ကို Lumini `.env` ထဲ ထည့်သွင်းပါ။

In [ ]:
# Step 1: Install Dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg ninja-build
!pip install -q voxcpm ninja fastapi uvicorn soundfile torch torchaudio python-multipart pycloudflared


In [ ]:
# Step 2: Launch VoxCPM2 and Official Cloudflare Tunnel
import os, subprocess, time, re, urllib.request, json

# 1. Clean up old processes & tunnel logs
os.system("killall -9 cloudflared 2>/dev/null; fuser -k 8000/tcp 2>/dev/null; pkill -9 -f 'python.*app.py' 2>/dev/null; rm -rf /root/.cloudflared /tmp/tunnel.log /tmp/app.log")
time.sleep(2)

server_code = r'''import os
import io
import sys
import uuid
import tempfile
from pathlib import Path
from typing import Optional
import soundfile as sf
import torch
import numpy as np
import re
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="Lumini VoxCPM2 Free GPU Engine")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

VOICE_PROMPTS_DIR = Path(tempfile.gettempdir()) / "lumini_voxcpm_prompts"
VOICE_PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

model = None
model_error = None
try:
    print("⏳ Loading OpenBMB VoxCPM2 Model into GPU...", flush=True)
    from voxcpm import VoxCPM
    model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
    print("✅ VoxCPM2 Model loaded successfully into GPU!", flush=True)
except Exception as e:
    import traceback
    traceback.print_exc()
    model_error = str(e)
    print(f"❌ Model load exception: {e}", flush=True)

@app.get("/api/voxcpm/status")
@app.get("/health")
async def get_status():
    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
    if model is None:
        return {
            "online": False,
            "ok": False,
            "error": model_error or "VoxCPM2 model is still downloading/initializing",
            "cuda": cuda_available,
            "device": device_name
        }
    return {
        "online": True,
        "ok": True,
        "engine": "VoxCPM2 (OpenBMB 48kHz High-Fidelity)",
        "cuda": cuda_available,
        "device": device_name,
        "sample_rate": 48000,
        "supports_zero_shot": True
    }

@app.post("/api/voxcpm/clone")
async def register_voice_sample(
    name: str = Form(...),
    instruction: str = Form("Energetic movie recap narration style"),
    transcript: str = Form(None),
    file: UploadFile = File(...)
):
    voice_id = str(uuid.uuid4())
    raw_tmp = VOICE_PROMPTS_DIR / f"raw_{voice_id}.tmp"
    target_path = VOICE_PROMPTS_DIR / f"{voice_id}.wav"
    
    content = await file.read()
    with open(raw_tmp, "wb") as f:
        f.write(content)

    # Convert to clean 24kHz 16-bit mono WAV using ffmpeg
    os.system(f"ffmpeg -y -i '{raw_tmp}' -ar 24000 -ac 1 -c:a pcm_s16le '{target_path}' >/dev/null 2>&1")
    if not target_path.exists() or target_path.stat().st_size < 100:
        with open(target_path, "wb") as f:
            f.write(content)

    return {
        "voiceId": voice_id,
        "name": name,
        "instruction": instruction,
        "message": "Voice sample registered and converted successfully."
    }

def split_burmese_text(text: str, max_chunk_len: int = 240) -> list[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    raw_sentences = re.split(r'(?<=[။!?\n])\s*', text)
    chunks = []
    current = ""
    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue
        if len(current) + len(s) + 1 <= max_chunk_len:
            current = (current + " " + s).strip() if current else s
        else:
            if current:
                chunks.append(current)
            if len(s) > max_chunk_len:
                sub_parts = re.split(r'(?<=[၊,])\s*', s)
                sub_curr = ""
                for sp in sub_parts:
                    sp = sp.strip()
                    if not sp:
                        continue
                    if len(sub_curr) + len(sp) + 1 <= max_chunk_len:
                        sub_curr = (sub_curr + " " + sp).strip() if sub_curr else sp
                    else:
                        if sub_curr:
                            chunks.append(sub_curr)
                        while len(sp) > max_chunk_len:
                            chunks.append(sp[:max_chunk_len])
                            sp = sp[max_chunk_len:]
                        sub_curr = sp
                current = sub_curr
            else:
                current = s
    if current:
        chunks.append(current)
    return chunks if chunks else [text]

@app.post("/api/voxcpm/synthesize")
async def synthesize_speech(
    voice_id: Optional[str] = Form(None),
    text: str = Form(...),
    instruction: Optional[str] = Form(None),
    speed: float = Form(1.0)
):
    if model is None:
        raise HTTPException(status_code=503, detail=f"VoxCPM model not ready: {model_error or 'Initializing'}")

    audio_path = None
    if voice_id and voice_id != 'default':
        for ext in [".wav", ".mp3", ".webm", ".m4a", ".ogg"]:
            candidate = VOICE_PROMPTS_DIR / f"{voice_id}{ext}"
            if candidate.exists():
                audio_path = candidate
                break

    try:
        chunks = split_burmese_text(text, max_chunk_len=240)
        print(f"🎙️ Synthesizing {len(chunks)} chunks with VoxCPM2...", flush=True)

        audio_pieces = []
        sample_rate = 48000
        if hasattr(model, 'tts_model') and hasattr(model.tts_model, 'sample_rate'):
            sample_rate = model.tts_model.sample_rate

        pause_samples = int(sample_rate * 0.08)
        pause_array = np.zeros(pause_samples, dtype=np.float32)

        for idx, chunk in enumerate(chunks):
            if not chunk.strip():
                continue

            if audio_path and os.path.exists(audio_path) and os.path.getsize(audio_path) > 100:
                wav_chunk = model.generate(
                    text=chunk,
                    reference_wav_path=str(audio_path),
                    cfg_value=2.0,
                    inference_timesteps=10
                )
            else:
                wav_chunk = model.generate(
                    text=chunk,
                    cfg_value=2.0,
                    inference_timesteps=10
                )

            if isinstance(wav_chunk, torch.Tensor):
                wav_chunk = wav_chunk.detach().cpu().numpy()
            
            if isinstance(wav_chunk, np.ndarray):
                wav_chunk = wav_chunk.squeeze()

            if hasattr(wav_chunk, 'dtype') and wav_chunk.dtype == np.float64:
                wav_chunk = wav_chunk.astype(np.float32)

            audio_pieces.append(wav_chunk)
            if idx < len(chunks) - 1:
                audio_pieces.append(pause_array)

        if not audio_pieces:
            raise HTTPException(status_code=400, detail="No audio chunks generated.")

        full_audio = np.concatenate(audio_pieces)
        max_val = np.max(np.abs(full_audio))
        if max_val > 0.01:
            full_audio = (full_audio / max_val) * 0.95

        buf = io.BytesIO()
        sf.write(buf, full_audio, sample_rate, format='WAV')
        buf.seek(0)
        return Response(content=buf.read(), media_type="audio/wav")
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8000)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

# 2. Install official cloudflared binary
if not os.path.exists("/usr/local/bin/cloudflared") and not os.path.exists("/usr/bin/cloudflared"):
    os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

# 3. Start FastAPI server with live unbuffered output and log capture
app_log = open('/tmp/app.log', 'w')
proc_app = subprocess.Popen(['python', '-u', 'app.py'], stdout=app_log, stderr=app_log)

# 4. Start official cloudflared tunnel
tunnel_log = open('/tmp/tunnel.log', 'w')
proc_tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=tunnel_log)

# 5. Extract Cloudflare URL
fresh_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists('/tmp/tunnel.log'):
        with open('/tmp/tunnel.log', 'r') as f:
            content = f.read()
            matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if matches:
                fresh_url = matches[-1]
                break

# 6. Healthcheck wait loop: Stream live progress & wait for GPU model to be 100% online
print("\n⏳ Downloading & loading VoxCPM2 weights into GPU (takes ~30-60s on first run)...")
is_ready = False
device_info = "GPU"
last_log_size = 0

for attempt in range(300):
    # Print any new lines from app.log to Colab console
    if os.path.exists('/tmp/app.log'):
        with open('/tmp/app.log', 'r') as f:
            lines = f.readlines()
            if len(lines) > last_log_size:
                for line in lines[last_log_size:]:
                    print(line.rstrip())
                last_log_size = len(lines)

    # Check if app crashed
    if proc_app.poll() is not None:
        print(f"\n❌ app.py exited prematurely with code {proc_app.returncode}! Full log:")
        if os.path.exists('/tmp/app.log'):
            with open('/tmp/app.log', 'r') as f:
                print(f.read())
        break

    try:
        req = urllib.request.urlopen("http://127.0.0.1:8000/api/voxcpm/status", timeout=2)
        if req.status == 200:
            st_data = json.loads(req.read().decode('utf-8'))
            if st_data.get('online'):
                is_ready = True
                device_info = st_data.get('device', 'GPU')
                break
    except Exception:
        pass
    time.sleep(1)

if is_ready and fresh_url:
    print("\n" + "="*70)
    print("🎉 🟢 VOXCPM2 FREE GPU SERVER IS 100% READY & ONLINE!")
    print(f"⚡ Device: {device_info} | Audio Quality: 48kHz Studio Zero-Shot")
    print("="*70)
    print(f"👉 Copy this URL and paste into Lumini (Voiceover -> VoxCPM2 button or .env):\n\n{fresh_url}\n")
    print("="*70 + "\n")
elif not is_ready:
    print(f"\n⚠️ Server initialization warning. URL: {fresh_url}. Check above logs.")

proc_app.wait()
